In [1]:
import pandas as pd
import sys
sys.path.append("../../lsr_package")
from lsr.transformer import LSR
from pyterrier_pisa import PisaIndex, PisaToksIndexer
from tqdm import tqdm
import random
import json
import pyterrier as pt
if not pt.started():
    pt.init()

/opt/miniconda3/envs/tmp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_15828/4287543803.py:10: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_15828/4287543803.py:11: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [2]:
model_path = "/nfs/primary/SPLADE/splade_sans/learned-sparse-retrieval-1.0.0/outputs/splade_max_strategy1_group8/model"

In [3]:
model = LSR(model_path)

In [2]:
full_df = pd.read_csv("/nfs/primary/SPLADE/splade_sans/data/training_data_w_qids.csv")

In [3]:
full_df

,medline_ui,title,abstract,sensitivity,text,querygen,querygen_score,best_query,qid
0,88000001,The binding of acetaldehyde to the active site...,"Ribonuclease A was reacted with [1-13C,1,2-14C...",0,The binding of acetaldehyde to the active site...,what site of activity is used to bind ribonucl...,[0.88141286 2.3967025 0.32945463 0.9057854 2...,does acetaldehyde react with ribonuclease A,0
1,88000002,Reductions in breath ethanol readings in norma...,Blood ethanol concentrations were measured seq...,0,Reductions in breath ethanol readings in norma...,is ethanol in breath normal\nis blood ethanol ...,[2.034942 3.5010455 0.3425358 0.5842041 0...,is blood ethanol measured sequentially for a n...,1
2,88000003,Does the blockade of opioid receptors influenc...,We have tested whether the opioid antagonists ...,0,Does the blockade of opioid receptors influenc...,which opioids can cause ethanol dependence?\ne...,[2.229129 1.2757981 2.7790852 0.18533784 2...,does the blockade of opioid receptors influenc...,2
3,88000006,Drinkwatchers--description of subjects and eva...,Clinical examination and measurement of MCV an...,0,Drinkwatchers--description of subjects and eva...,heavy drinking blood test\nhow much alcohol is...,[0.87555873 0.48380333 1.1801778 0.89115256 1...,how much alcohol do drinkwatchers drink,3
4,88000007,Platelet affinity for serotonin is increased i...,The kinetics of 3H serotonin platelet uptake w...,0,Platelet affinity for serotonin is increased i...,what is the biological marker for dependence\n...,[3.0765808 1.4992186 1.945047 1.81894 2...,does alcoholics use serotonin to increase plat...,4
...,...,...,...,...,...,...,...,...,...
334065,87325749,Clinical investigation of Duchenne muscular dy...,Between 1979 and 1987 we documented the natura...,0,Clinical investigation of Duchenne muscular dy...,what is clinical investigation of duchenne mus...,[4.7701135 3.1781783 3.1509128 3.3405652 3...,what is clinical investigation of duchenne mus...,334129
334066,87325791,Prostaglandins. Old concepts and new perspecti...,Lupus anticoagulants and anticardiolipin antib...,0,Prostaglandins. Old concepts and new perspecti...,what prostaglandins do\nprostaglandins lupus\n...,[0.8321806 1.5327109 3.6516025 2.132977 0...,what is a lupus anticoagulant and anticardioli...,334130
334067,87325811,Vinculin in focal cell-to-substrate attachment...,Vinculin is a specialized protein with a molec...,0,Vinculin in focal cell-to-substrate attachment...,what is the function of vinculin\nwhat is vinc...,[1.5229188 3.623153 3.2570174 2.759778 1...,where does vinculin attach in cells to a subst...,334131
334068,87325849,Intraocular fibrinolysis with recombinant huma...,Intraocular fibrin formation following ocular ...,0,Intraocular fibrinolysis with recombinant huma...,recombinant plasminogen activator treatment\ni...,[1.6216362 3.4744203 3.2590685 1.8076218 0...,intraocular fibrinolysis with recombinant plas...,334132


In [13]:
qrels_dict = full_df.groupby(full_df["qid"].astype(str))["medline_ui"] \
               .apply(lambda x: list(map(str, x))) \
               .to_dict()

with open("./score_files/qrels.json", "w") as f:
    json.dump(qrels_dict, f, indent=4)

In [49]:
# lookup_dict = dict(zip(full_df["medline_ui"], full_df["text"]))
# qid2docno = dict(zip(full_df["qid"], full_df["medline_ui"]))
qid2query = dict(zip(query_frame["qid"], query_frame["query"]))

In [6]:
def iterator():
    for _, doc in tqdm(full_df.iterrows(), total = len(full_df), desc = "indexing"):
        yield {"docno" : str(doc.medline_ui), "text" : doc.text}

# Index the corpus
# print("Starting indexing...")
# index_pipeline = model >> PisaToksIndexer("./indices/sans1_training_index")
# index = index_pipeline.index(iterator())
# print("Indexing completed!")

index = PisaIndex("./indices/sans1_training_index")

In [7]:
def filter(run):
    original_len = len(run)
    full_df.medline_ui = full_df.medline_ui.astype(str)
    run = pd.merge(run, full_df, left_on = "docno", right_on = "medline_ui")
    assert len(run) == original_len, "merge error"
    run = run[run["sensitivity"] == 1]
    run = run.drop(columns= ["rank"])
    run = pt.model.add_ranks(run)
    full_df.medline_ui = full_df.medline_ui.astype(int)
    return run

In [32]:
retr = model.query_encoder() >> index.quantized() >> pt.apply.generic(filter) % 50
retr(pd.DataFrame(query_frame.iloc[24]).T)

,qid,query,query_toks,docno,score,medline_ui,title,abstract,sensitivity,text,querygen,querygen_score,best_query,rank
0,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",88000323,87168.0,88000323,"Atracurium, vecuronium and pancuronium in end-...","Dose-response relations for atracurium, vecuro...",1,"Atracurium, vecuronium and pancuronium in end-...",what is the use of vecuronium in end stage ren...,[1.0224634 0.61069334 2.2256982 0.19395651 0...,what is the dose-response interaction of atrac...,0
1,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",88000304,64587.0,88000304,Vecuronium and atracurium in patients with end...,"Twenty patients with end-stage renal failure, ...",1,Vecuronium and atracurium in patients with end...,what is the difference between vecuronium and ...,[3.2097313 3.2097313 3.2097313 1.0351628 3.213...,vecuronium and atracurium for treatment of end...,1
2,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",88131713,60652.0,88131713,Pancuronium and renal failure [letter],There are technical difficulties associated wi...,1,Pancuronium and renal failure [letter] There a...,can pancuronium cause renal failure\nwhat is p...,[1.8139415 1.2343276 0.7475966 1.8139415 1...,can pancuronium help renal failure,2
3,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",87049478,57840.0,87049478,Use of atracurium in pre-eclamptic patients [l...,"Double-lumen endobronchial tubes were placed ""...",1,Use of atracurium in pre-eclamptic patients [l...,what is the use of the atracurium\nwhich tube ...,[1.7894913 2.9651167 1.8458165 3.3247313 2...,atracurium use in preeclamptic patients,3
4,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",90067360,53953.0,90067360,Intravenous pancuronium bromide for fetal neur...,Intravenous pancuronium bromide was administer...,1,Intravenous pancuronium bromide for fetal neur...,intravenous pancuronium bromide for fetus\nwha...,[4.190875 3.8539617 3.6987634 3.188139 3...,pancuronium bromide for fetal neuromuscular bl...,4
5,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",87271396,50350.0,87271396,Pharmacokinetics of atracurium and its metabol...,The plasma pharmacokinetic profiles of atracur...,1,Pharmacokinetics of atracurium and its metabol...,what metabolite is atracurium\ndoes atracurium...,[1.6842266 2.6817095 0.23122434 0.97524387 3...,atracurium pharmacokinetics,5
6,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",90379526,49983.0,90379526,Atracurium in a patient with acute intermitten...,The efficiencies of airway heat and moisture e...,1,Atracurium in a patient with acute intermitten...,is atranurium anesthesia\nwhich filter is an a...,[0.22076638 1.2387491 1.38586 2.1968482 0...,which airway heat and moisture exchanging filt...,6
7,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",88047546,49630.0,88047546,Neuromuscular interaction of magnesium with su...,Stability of halogenated volatile anesthetic i...,1,Neuromuscular interaction of magnesium with su...,what is the neuromuscular interaction of magne...,[1.8555138 0.54771113 2.660082 0.44641393 0...,neuromuscular interaction of magnesium and suc...,7
8,24,what is the dose-response interaction of atrac...,"{'dose': 1.550574779510498, 'pan': 1.389566183...",90003559,49558.0,90003559,Should vecuronium be used in renal failure? [l...,We report the successful treatment with esmolo...,1,Should vecuronium be used in renal failure? [l...,vecuronium should be used when\nwhat is the tr...,[2.453699 1.5336199 3.7234805 2.3208258 0...,can vecuronium be used in renal failure?,8
9,24,what is the dose-response intera

In [35]:
qid2docno[24]

88000323

In [31]:
# def is_sensitive(docnos):
#     full_df.medline_ui = full_df.medline_ui.astype(str)
#     filtered_df = full_df[full_df['medline_ui'].isin(docnos)].sensitivity.sum()
#     full_df.medline_ui = full_df.medline_ui.astype(int)
#     return filtered_df 



def safe_sample(docnos, k):
    if len(docnos) >= k:
        return random.sample(docnos, k)
    else:
        return random.choices(docnos, k=k)

full_df = full_df.drop(columns = ["qid"])

In [33]:
# retrieval_results = retr(query_frame.head(100))
# retrieval_results.to_csv("./hn_mining_retrieval_results.csv.gz", index = False)

In [ ]:
# output_list = []

# retr = model.query_encoder() >> index.quantized() >> pt.apply.generic(filter) % 50
# for _, doc in tqdm(full_df.iterrows(), total = len(full_df), desc = "negative mining"):
#     res = retr.search(str(doc.best_query))
#     candidate_docnos = res.docno.tolist()
#     filtered_docnos = [x for x in candidate_docnos if x != str(doc.medline_ui)]
#     selected_docnos = safe_sample(filtered_docnos, k=8)
#     output_list.append({"query" : str(doc.best_query), "docno_a" : lookup_dict[doc.medline_ui], "docno_b" : [lookup_dict[int(docno)] for docno in selected_docnos]})

In [ ]:
# with open("hn_splade_sans1_all_sensitive.jsonl", "w") as f:
#     for item in output_list:
#         f.write(json.dumps(item) + "\n")

In [ ]:
# file_path = "hn_splade_sans1.jsonl"
# OUTPUTS = ["true", "false"]

# def iter_jsonl_with_index(filepath):
#     with open(filepath, "r") as f:
#         for i, line in enumerate(f):
#             data = json.loads(line)
#             yield 'Query: ' + data["query"] + ' Document: ' + data["docno_a"] + ' Relevant:', OUTPUTS[0]
#             for neg in data["docno_b"]:
#                 yield 'Query: ' + data["query"] + ' Document: ' + neg + ' Relevant:', OUTPUTS[1]
 
# iter_jsonl_with_index(file_path)

In [37]:
retrieval_results = pd.read_csv("hn_mining_retrieval_results.csv.gz")

In [59]:
output_list = []
docnos_list = []

for _, res in tqdm(retrieval_results.groupby("qid"), total = retrieval_results.qid.nunique(), desc = "processing query"):
    candidate_docnos = res.docno.tolist()
    qid = res.qid.iloc[0]
    filtered_docnos = [x for x in candidate_docnos if x != qid2docno[qid]]
    selected_docnos = safe_sample(filtered_docnos, k=8)
    pos_docno = qid2docno[qid]
    query = qid2query[qid]
    output_list.append({"query" : query, "docno_a" : lookup_dict[pos_docno], "docno_b" : [lookup_dict[int(d)] for d in selected_docnos]})
    docnos_list.append({"query" : int(qid), "docno_a" : pos_docno, "docno_b" : [d for d in selected_docnos]})

processing query: 100%|██████████| 334065/334065 [00:26<00:00, 12442.84it/s]


In [60]:
with open("hn_splade_sans1_all_sensitive_8_negs_v2.jsonl", "w") as f:
    for item in output_list:
        f.write(json.dumps(item) + "\n")

with open("hn_splade_sans1_all_sensitive_8_negs_docnos_v2.jsonl", "w") as f:
    for item in docnos_list:
        f.write(json.dumps(item) + "\n")

In [18]:
import json
with open("/nfs/primary/SPLADE/splade_class/knowledge_distillation/rq2/score_files/hn_t5base_8negs_scorefile.json", "r") as f:
    scores = json.load(f)
    
with open("/nfs/primary/SPLADE/splade_class/knowledge_distillation/rq2/score_files/qrels.json", "r") as f:
    qrels = json.load(f)

In [19]:
qrels["64040"]

['88156921']

KeyError: '64040'

In [21]:
import pandas as pd
data = pd.read_csv("/nfs/primary/SPLADE/splade_class/knowledge_distillation/rq2/score_files/hn_t5base_8negs_scoredf.csv")

In [37]:
full_df[full_df["qid"] == 150013]

,medline_ui,title,abstract,sensitivity,text,querygen,querygen_score,best_query,qid
149984,90086816,Rebound surges of intracranial pressure as a c...,Cerebral edema remains the most common immedia...,0,Rebound surges of intracranial pressure as a c...,which is most common cause of death during ful...,[4.033416 3.9259207 3.4138124 3.7209785 1...,which is most common cause of death during ful...,150013
